In [ ]:
!pip install -q transformers peft datasets accelerate

In [ ]:
!pip install --upgrade torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 20.4 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [ ]:
import sys
import os
from google.colab import drive

# 1. Monta il Drive
drive.mount('/content/drive')

# 2. Definisci i percorsi principali
BASE_DRIVE = '/content/drive/MyDrive/DeepLearning'
SRC_DIR = os.path.join(BASE_DRIVE, 'src')

# 3. Aggiungi la cartella 'src' ai percorsi di sistema di Python
if SRC_DIR not in sys.path:
    sys.path.append(SRC_DIR)
    print(f"Directory {SRC_DIR} aggiunta al path di sistema.")

# 4. Ora puoi importare normalmente come se fossero librerie installate
from dataset import CLEVRDataset
from models import MultimodalCoT

import torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer
from peft import LoraConfig, get_peft_model
from tqdm import tqdm

# Impostiamo il device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Setup completato. Device in uso: {device}")

Mounted at /content/drive
Directory /content/drive/MyDrive/DeepLearning/src aggiunta al path di sistema.


✅ Setup completato. Device in uso: cuda


In [ ]:
from torchvision import transforms

# 1. Inizializziamo il Tokenizer del T5 (o del modello LLM base che stai usando)
MODEL_NAME = "t5-base" # Cambialo se usi t5-base o un altro modello
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# 2. Definiamo la pipeline di trasformazione per le immagini
# (Usa quella esatta che richiede il tuo vision_encoder)
transform = transforms.Compose([
    transforms.Resize((384, 384)), # Adatta alla dimensione richiesta dal tuo encoder
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

print("✅ Tokenizer e Transform pronti.")

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

✅ Tokenizer e Transform pronti.


In [ ]:
# Percorsi assoluti verso il tuo ecosistema "blindato"
BASE_DRIVE = '/content/drive/MyDrive/DeepLearning'
TRAIN_INDEX = os.path.join(BASE_DRIVE, 'indexes/train_index.json')
TRAIN_IMG_DIR = os.path.join(BASE_DRIVE, 'data/processed/images/train')

# Inizializziamo il Dataset
train_dataset = CLEVRDataset(
    index_path=TRAIN_INDEX,
    img_dir=TRAIN_IMG_DIR,
    tokenizer=tokenizer,
    transform=transform,
    stage='stage1'
)

# --- CREIAMO IL CUSTOM COLLATOR ---
def custom_collate_fn(batch):
    import torch
    return {
        # Impila le immagini in un unico tensore [batch_size, channels, height, width]
        'pixel_values': torch.stack([item['pixel_values'] for item in batch]),

        # Lascia i testi come semplici liste di stringhe (saranno processati dal tokenizer nella Cella 5)
        'input_text': [item['input_text'] for item in batch],
        'target_text': [item['target_text'] for item in batch],

        # Lascia i dati grezzi come semplice lista di dizionari (ignorati da PyTorch)
        'raw_item': [item['raw_item'] for item in batch]
    }

# Creiamo il DataLoader inserendo la nostra funzione personalizzata
BATCH_SIZE = 8
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True,
    collate_fn=custom_collate_fn # <--- Il fix è qui!
)

print(f"✅ DataLoader pronto. Campioni totali: {len(train_dataset)} | Batch totali: {len(train_loader)}")

✅ DataLoader pronto. Campioni totali: 15000 | Batch totali: 1875


In [ ]:
# 1. Inizializza l'architettura completa
model = MultimodalCoT().to(device)

# 2. Configurazione LoRA per il modello linguistico interno
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q", "v"], # Verifica che questi siano i layer corretti per T5
    lora_dropout=0.05,
    bias="none",
    task_type="SEQ_2_SEQ_LM"
)

# 3. Innestiamo LoRA nel sottomodello LLM
model.llm = get_peft_model(model.llm, lora_config)

# Stampiamo i parametri addestrabili per conferma
model.llm.print_trainable_parameters()

model.safetensors:   0%|          | 0.00/1.23G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/892M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

trainable params: 884,736 || all params: 223,788,288 || trainable%: 0.3953


In [ ]:
from torch.optim import AdamW
import torch.amp

# IPERPARAMETRI OTTIMIZZATI
EPOCHS = 5
LEARNING_RATE = 5e-4
ACCUMULATION_STEPS = 4 # Aggiorna i pesi ogni 4 batch (simula una batch size 4 volte più grande)

optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)
scaler = torch.amp.GradScaler('cuda') # Il "Motore" per la velocità a 16-bit

print("🚀 Inizio Addestramento (con Accelerazione AMP e Cross-Attention)...")
model.train()

for epoch in range(EPOCHS):
    total_loss = 0
    optimizer.zero_grad() # Resettiamo all'inizio dell'epoca

    progress_bar = tqdm(train_loader, desc=f"Epoca {epoch+1}/{EPOCHS}")

    for step, batch in enumerate(progress_bar):
        pixel_values = batch['pixel_values'].to(device)

        # ADDIO PADDING A 128! Ora usiamo il padding dinamico pulito
        inputs = tokenizer(batch['input_text'], return_tensors="pt", padding=True, truncation=True).to(device)
        targets = tokenizer(batch['target_text'], return_tensors="pt", padding=True, truncation=True, max_length=256).to(device)

        labels = targets.input_ids
        labels[labels == tokenizer.pad_token_id] = -100

        # --- FORWARD PASS VELOCE (MIXED PRECISION) ---
        # torch.autocast permette alla GPU di usare i Tensor Cores per fare calcoli alla velocità della luce
        with torch.amp.autocast('cuda'):
            outputs = model(
                pixel_values=pixel_values,
                input_ids=inputs.input_ids,
                attention_mask=inputs.attention_mask,
                labels=labels
            )
            # Dividiamo la loss per i passi di accumulo
            loss = outputs.loss / ACCUMULATION_STEPS

        # --- BACKWARD PASS SCALATO ---
        scaler.scale(loss).backward()

        # Aggiorniamo i pesi solo ogni 'ACCUMULATION_STEPS'
        if (step + 1) % ACCUMULATION_STEPS == 0 or (step + 1) == len(train_loader):
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        # Moltiplichiamo per ACCUMULATION_STEPS solo per stampare il valore reale della loss a schermo
        total_loss += (loss.item() * ACCUMULATION_STEPS)
        progress_bar.set_postfix({'loss': f"{(loss.item() * ACCUMULATION_STEPS):.4f}"})

    avg_loss = total_loss / len(train_loader)
    print(f"🏁 Fine Epoca {epoch+1} | Loss Media: {avg_loss:.4f}\n")

🚀 Inizio Addestramento (con Accelerazione AMP e Cross-Attention)...


Epoca 1/5: 100%|██████████| 1875/1875 [1:10:53<00:00,  2.27s/it, loss=0.1586]


🏁 Fine Epoca 1 | Loss Media: 0.7980



Epoca 2/5: 100%|██████████| 1875/1875 [11:53<00:00,  2.63it/s, loss=0.0323]


🏁 Fine Epoca 2 | Loss Media: 0.0707



...

🏁 Fine Epoca 3 | Loss Media: 0.0312



Epoca 4/5: 100%|██████████| 1875/1875 [11:41<00:00,  2.67it/s, loss=0.0174]


🏁 Fine Epoca 4 | Loss Media: 0.0230



Epoca 5/5: 100%|██████████| 1875/1875 [11:39<00:00,  2.68it/s, loss=0.0441]

🏁 Fine Epoca 5 | Loss Media: 0.0203



In [ ]:
# Definiamo la cartella di output finale
ADAPTER_DIR = os.path.join(BASE_DRIVE, 'lora_adapter_final')
os.makedirs(ADAPTER_DIR, exist_ok=True)

# 1. Salviamo il cervello linguistico (LoRA e Tokenizer)
model.llm.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

# 2. SALVIAMO GLI OCCHI E IL RAGIONAMENTO (I moduli Custom)
custom_weights = {
    'projector': model.projector.state_dict(),
    'cross_attention': model.cross_attention.state_dict(),
    'layer_norm': model.layer_norm.state_dict()
}
torch.save(custom_weights, os.path.join(ADAPTER_DIR, 'custom_modules.pth'))

print(f"🎉 Modello e moduli visivi salvati con successo in: {ADAPTER_DIR}")

🎉 Modello e moduli visivi salvati con successo in: /content/drive/MyDrive/DeepLearning/lora_adapter_final
